# Block 3 Exercise — Validate and Check

**Goal:** run odmlib's four validation layers, interpret what each one finds, and repair a
broken define.xml until it validates clean.

**Time budget:** ~20 minutes for TODOs 1–4. The Stretch section is optional.

The working cells first run every layer against a known-clean file so you see what "all
green" looks like. Then you get broken files.

In [1]:
import os
import odmlib.define_loader as DL
import odmlib.loader as LD
from odmlib.odm_parser import ODMSchemaValidator
from odmlib import create_oid_checker
from odmlib.define_2_1.rules.metadata_schema import MetadataSchema

os.makedirs("output", exist_ok=True)

def load_define(path):
    """Load a Define-XML v2.1 file and return the ODM root object."""
    loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
    loader.open_odm_document(path)
    return loader.root()      # one root() call - keep this reference for any edits

xsd = ODMSchemaValidator(standard="define", version="2.1")
print("ready")

ready


### All four layers on a clean file

The MSG example define.xml from Block 1 passes everything. Layer 1 — XSD schema validation
(file-level, using the schema bundled with odmlib):

In [2]:
xsd.validate_file("../data/defineV21-SDTM.xml")
print("layer 1 (XSD):         PASS")

layer 1 (XSD):         PASS


Layers 2–4 work on the object tree. Individually:

In [3]:
odm = load_define("../data/defineV21-SDTM.xml")

odm.verify_oids(create_oid_checker("define_2_1"))
print("layer 2 (OID ref/def): PASS")

odm.verify_conformance(MetadataSchema())
print("layer 3 (conformance): PASS")

odm.verify_order()
print("layer 4 (order):       PASS")

layer 2 (OID ref/def): PASS
layer 3 (conformance): PASS
layer 4 (order):       PASS


And the combined call — the everyday default. `collect_errors=True` returns **every** finding
across layers 2–4 as a list; empty means clean. Note the fresh OID checker: checkers are
stateful and single-use per document.

In [4]:
errors = odm.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"combined validate: {len(errors)} errors")

combined validate: 0 errors


## TODO 1 — Diagnose a schema-invalid file

`data/defineV21-SDTM-invalid.xml` fails XSD validation. Use `xsd.xsd.iter_errors(path)` to
enumerate the findings and print each error's `reason` and `path`.

**Question:** which required attribute is missing, and from which element?

*Hint (lecture §1):* `iter_errors` yields error objects; `err.reason` and `err.path` tell the
story.

In [ ]:
# YOUR CODE HERE: enumerate the XSD errors in ../data/defineV21-SDTM-invalid.xml
# and print reason + path for each

## TODO 2 — Find the reference errors

`data/define_broken_refs.xml` is schema-valid — but broken. Load it with `load_define()` and
run `verify_oids` inside a `try/except OdmlibOIDError` block, printing the error.

**Question:** which OID is referenced but never defined?

*Hint (lecture §2):* to XSD, an OID is just a string — that's why this file passes layer 1
and fails layer 2.

In [ ]:
from odmlib.exceptions import OdmlibOIDError

broken = load_define("../data/define_broken_refs.xml")
mdv = broken.Study.MetaDataVersion

# YOUR CODE HERE: run broken.verify_oids(...) in a try/except and print the error

## TODO 3 — Repair it until it validates clean

The dangling reference is a typo: an `ItemRef` points at `IT.DM.SEXX`, but the variable is
defined as `IT.DM.SEX`. Repair the document **as objects** — no text editing:

1. Find the bad `ItemRef` (it's in the first ItemGroupDef) and set its `ItemOID` to
   `"IT.DM.SEX"`.
2. Re-run `verify_oids` — it should pass now.
3. Now that references are clean, call `broken.unreferenced_oids(...)` — it reports an
   **orphan**: an ItemDef defined but never referenced. Find it with `mdv.find()` and remove
   it from `mdv.ItemDef` (`.remove(...)`).
4. Run the combined `validate(collect_errors=True, ...)` — expect an empty list.
5. Write the repaired file to `output/define_repaired.xml` and XSD-validate it.

*Hint (lecture §2):* `unreferenced_oids` raises while dangling refs remain — that's why the
fix order is refs first, orphans second. And remember: fresh checker for every call.

In [ ]:
# YOUR CODE HERE: step 1 - fix the ItemRef typo, step 2 - verify_oids

# YOUR CODE HERE: step 3 - find and remove the orphan ItemDef

# YOUR CODE HERE: steps 4-5 - combined validate to an empty list, write, XSD-validate

## TODO 4 — Why you need both the object checks and XSD

`data/defineV21-SDTM-invalid-class.xml` contains a `def:Class` with `Name="NONSENSE"` — not a
valid dataset class. Run **both**:

1. Load it and run the combined `validate(collect_errors=True, ...)` on the objects.
2. Run `xsd.xsd.iter_errors(...)` on the file and print each `reason`.

**Question:** which layer catches the bad class name — and what does that tell you about
relying on a single layer?

In [ ]:
# YOUR CODE HERE: 1) combined validate on the loaded objects, 2) XSD iter_errors on the file

## Stretch — permissive-mode repair

`data/nonconformant_define21.xml` is so broken it won't even load strictly (an ItemGroupDef
is missing its required `Repeating` attribute). This is the full repair loop from lecture §6:
load permissively, collect every error, fix the objects, re-validate to clean.

Run the first cell to see the strict failure and the collected findings:

In [ ]:
import odmlib
from odmlib.exceptions import OdmlibRequiredAttributeError

try:
    load_define("../data/nonconformant_define21.xml")
except OdmlibRequiredAttributeError as e:
    print("strict load fails:", e)

with odmlib.permissive():
    nc = load_define("../data/nonconformant_define21.xml")
print("\npermissive load OK")

errors = nc.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
for err in errors:
    print(type(err).__name__, "-", str(err).splitlines()[0])

Now fix both findings on the object tree — set the missing `Repeating` to `"No"` on the first
`ItemGroupDef`, and change the bogus `Origin` `Type` on the first `ItemDef` to `"Collected"` —
then re-run the combined validate and confirm it comes back empty.

In [ ]:
nc_mdv = nc.Study.MetaDataVersion

# YOUR CODE HERE: repair the two findings, then re-run validate to an empty list

**Done?** Compare with `solutions/validate_check_solution.ipynb`.

The loop you just ran — load (permissively if needed) → collect every error → fix objects →
re-validate to clean → write — is the core odmlib workflow, and the foundation of every
Define-XML pipeline and repair tool built on it.